# BrandSphere AI — Week 2: Logo CNN Classifier & Embedding Engine
**CRS AI Capstone 2025-26 | Scenario 1**

This notebook:
1. Downloads Logo Dataset from Kaggle (2341 classes, 167k images)
2. Preprocesses images (resize, normalize, augment)
3. Trains CNN classifier (Conv2D → MaxPooling → Flatten → Dense → Softmax)
4. Extracts embeddings from intermediate layer
5. Saves embeddings + model for Streamlit app retrieval

## Cell 1 — Install Dependencies

In [ ]:
!pip install kagglehub tensorflow opencv-python-headless scikit-learn matplotlib seaborn numpy pillow -q
print('✅ Dependencies installed')

## Cell 2 — Download Logo Dataset from Kaggle

In [ ]:
import kagglehub
import os

# Download Logo Dataset (2341 classes, 167140 images)
print('📥 Downloading Logo Dataset... (this may take a few minutes)')
logo_path = kagglehub.dataset_download('siddharthkumarsah/logo-dataset-2341-classes-and-167140-images')
print(f'✅ Logo Dataset downloaded to: {logo_path}')

# Explore structure
print('\n📁 Dataset structure:')
for root, dirs, files in os.walk(logo_path):
    level = root.replace(logo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files[:3]:
            print(f'{subindent}{f}')
    if level == 1 and len(dirs) > 5:
        print(f'{indent}  ... ({len(dirs)} total classes)')
        break

## Cell 3 — Find Dataset Root & Count Classes

In [ ]:
import os

# Find the actual image directory
dataset_root = None
for root, dirs, files in os.walk(logo_path):
    img_files = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if len(img_files) > 10 and len(dirs) > 10:
        dataset_root = root
        break
    elif len(dirs) > 50:  # likely the class-level directory
        dataset_root = root
        break

if dataset_root is None:
    dataset_root = logo_path

print(f'📂 Using dataset root: {dataset_root}')

# Count classes and images
classes = sorted([d for d in os.listdir(dataset_root)
                  if os.path.isdir(os.path.join(dataset_root, d))])
print(f'📊 Total classes found: {len(classes)}')
print(f'📋 Sample classes: {classes[:10]}')

# Count total images
total_images = 0
class_counts = {}
for cls in classes:
    cls_path = os.path.join(dataset_root, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    class_counts[cls] = len(imgs)
    total_images += len(imgs)
print(f'🖼️  Total images: {total_images}')

## Cell 4 — Select Top 50 Classes (for training efficiency)

In [ ]:
import numpy as np

# Use top 50 classes by image count for training
# (full 2341 classes would take too long on Colab)
TOP_N_CLASSES = 50
IMG_SIZE = 128
BATCH_SIZE = 32

top_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:TOP_N_CLASSES]
selected_classes = [c[0] for c in top_classes]
print(f'✅ Selected top {TOP_N_CLASSES} classes for training:')
for i, (cls, count) in enumerate(top_classes[:10]):
    print(f'  {i+1:2d}. {cls}: {count} images')
print(f'  ...')
print(f'\n📊 Total training images: {sum(c[1] for c in top_classes)}')

## Cell 5 — Preprocess Images (Resize, Normalize, Augment)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random

print('🔄 Loading and preprocessing images...')

X, y, image_paths = [], [], []
label_map = {cls: idx for idx, cls in enumerate(selected_classes)}

MAX_PER_CLASS = 30  # cap per class to keep Colab from running out of RAM

for cls in selected_classes:
    cls_dir = os.path.join(dataset_root, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    imgs = random.sample(imgs, min(MAX_PER_CLASS, len(imgs)))
    for img_file in imgs:
        img_path = os.path.join(cls_dir, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.astype(np.float32) / 255.0  # normalize 0-1
        X.append(img)
        y.append(label_map[cls])
        image_paths.append(img_path)

X = np.array(X)
y = np.array(y)
print(f'✅ Loaded {len(X)} images across {TOP_N_CLASSES} classes')
print(f'📐 Image shape: {X.shape}')

# Train/Val/Test split — 80/10/10
X_train, X_temp, y_train, y_temp, paths_train, paths_temp = train_test_split(
    X, y, image_paths, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test, paths_val, paths_test = train_test_split(
    X_temp, y_temp, paths_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f'\n📊 Split: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')

## Cell 6 — Visualize Sample Logos

In [ ]:
import matplotlib.pyplot as plt

reverse_map = {v: k for k, v in label_map.items()}

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle('Sample Logo Images from Dataset', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(X_train):
        ax.imshow(X_train[i])
        ax.set_title(reverse_map[y_train[i]][:12], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_logos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sample logos visualized')

## Cell 7 — Data Augmentation

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation for training set
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    fill_mode='nearest'
)

# No augmentation for val/test
val_datagen = ImageDataGenerator()

train_gen = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)
val_gen   = val_datagen.flow(X_val,   y_val,   batch_size=BATCH_SIZE)

print('✅ Data augmentation configured')
print(f'   Rotation: ±15° | Flip: horizontal | Brightness: 0.8-1.2x | Zoom: 10%')

## Cell 8 — Build CNN Model

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                      Dense, Dropout, BatchNormalization, Input)
from tensorflow.keras.optimizers import Adam

NUM_CLASSES = TOP_N_CLASSES

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Block 1
x = Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
x = BatchNormalization()(x)
x = MaxPooling2D(2,2)(x)

# Block 2
x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(2,2)(x)

# Block 3
x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(2,2)(x)

# Block 4
x = Conv2D(256, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(2,2)(x)

# Embedding layer (used for similarity search)
x = Flatten()(x)
embedding = Dense(512, activation='relu', name='embedding')(x)
x = Dropout(0.4)(embedding)

# Classification head
outputs = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f'\n✅ CNN built — {model.count_params():,} parameters')

## Cell 9 — Train CNN

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, monitor='val_loss'),
    ModelCheckpoint('best_logo_cnn.h5', save_best_only=True, monitor='val_accuracy')
]

print('🚀 Training CNN...')
history = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)
print('\n✅ Training complete!')

## Cell 10 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train Acc', color='#C9A84C', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Acc',   color='#3ECFB2', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'],     label='Train Loss', color='#C9A84C', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val Loss',   color='#3ECFB2', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('BrandSphere CNN Logo Classifier — Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Final accuracy
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'\n📊 Final Test Accuracy: {test_acc*100:.2f}%')
print(f'📊 Final Test Loss:     {test_loss:.4f}')

## Cell 11 — Extract Embeddings for Similarity Search

In [ ]:
# Build embedding extractor — outputs 512-dim vector per image
embedding_model = Model(inputs=model.input,
                        outputs=model.get_layer('embedding').output)

print('🔄 Extracting embeddings from ALL training images...')

# Extract embeddings for entire training set
all_embeddings = embedding_model.predict(X_train, batch_size=64, verbose=1)

print(f'✅ Embeddings shape: {all_embeddings.shape}')
print(f'   {len(all_embeddings)} images × 512 dimensions')

# Save everything
np.save('logo_embeddings.npy', all_embeddings)
np.save('logo_labels.npy', y_train)
np.save('logo_images.npy', X_train)

# Save class names
import json
with open('logo_classes.json', 'w') as f:
    json.dump({'label_map': label_map, 'classes': selected_classes}, f)

print('\n✅ Saved:')
print('   logo_embeddings.npy — 512-dim vectors for similarity search')
print('   logo_labels.npy     — class labels')
print('   logo_images.npy     — preprocessed images')
print('   logo_classes.json   — class name mapping')

## Cell 12 — Similarity Search Function (Top 5 Logos)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_logos(query_class_name: str, top_n: int = 5):
    """
    Given a brand class name, find the top N most visually similar logos.
    This is what the Streamlit app calls.
    """
    if query_class_name not in label_map:
        # Find closest matching class
        query_class_name = selected_classes[0]

    query_label = label_map[query_class_name]

    # Get mean embedding for this class
    class_mask = y_train == query_label
    if class_mask.sum() == 0:
        class_mask = np.ones(len(y_train), dtype=bool)
    query_embedding = all_embeddings[class_mask].mean(axis=0, keepdims=True)

    # Cosine similarity across all embeddings
    sims = cosine_similarity(query_embedding, all_embeddings)[0]

    # Top N most similar (excluding same class to get variety)
    top_indices = np.argsort(sims)[::-1][:top_n * 3]
    seen_classes = set()
    result_indices = []
    for idx in top_indices:
        cls = reverse_map[y_train[idx]]
        if cls not in seen_classes:
            seen_classes.add(cls)
            result_indices.append(idx)
        if len(result_indices) == top_n:
            break

    return [(X_train[i], reverse_map[y_train[i]], sims[i]) for i in result_indices]


# Test it
test_class = selected_classes[0]
results = find_similar_logos(test_class, top_n=5)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle(f'Top 5 Similar Logos for: {test_class}', fontsize=14, fontweight='bold')
for i, (img, cls, score) in enumerate(results):
    axes[i].imshow(img)
    axes[i].set_title(f'{cls[:14]}\n{score:.3f}', fontsize=9)
    axes[i].axis('off')
plt.tight_layout()
plt.savefig('top5_similar_logos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Similarity search working!')

## Cell 13 — PCA Cluster Visualization

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

# Reduce to 2D for visualization
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(all_embeddings)

# Plot first 15 classes only (for readability)
mask = y_train < 15
plt.figure(figsize=(14, 10))
scatter = plt.scatter(
    embeddings_2d[mask, 0], embeddings_2d[mask, 1],
    c=y_train[mask], cmap='tab20', alpha=0.7, s=40
)
plt.colorbar(scatter, label='Class Index')
plt.title('PCA of Logo Embeddings — Class Clusters', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ PCA cluster plot saved')

## Cell 14 — Save Model & Upload to Google Drive

In [ ]:
import pickle, shutil
from google.colab import drive

# Save full model
model.save('logo_cnn_model.h5')
embedding_model.save('logo_embedding_model.h5')

# Save everything in one pickle for Streamlit
streamlit_package = {
    'embeddings':     all_embeddings,
    'labels':         y_train,
    'images':         X_train,
    'label_map':      label_map,
    'reverse_map':    reverse_map,
    'selected_classes': selected_classes,
    'img_size':       IMG_SIZE,
}
with open('logo_retrieval_package.pkl', 'wb') as f:
    pickle.dump(streamlit_package, f)

print('✅ Saved locally:')
print('   logo_cnn_model.h5          — full CNN model')
print('   logo_embedding_model.h5    — embedding extractor')
print('   logo_retrieval_package.pkl — everything for Streamlit')

# Mount Drive and upload
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/BrandSphere_Models'
os.makedirs(save_dir, exist_ok=True)

for fname in ['logo_cnn_model.h5', 'logo_embedding_model.h5',
              'logo_retrieval_package.pkl', 'logo_embeddings.npy',
              'logo_classes.json', 'training_curves.png',
              'pca_clusters.png', 'top5_similar_logos.png']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(save_dir, fname))
        print(f'   ✅ Uploaded {fname} to Drive')

print(f'\n🎉 All files saved to Google Drive: {save_dir}')

## Cell 15 — Summary Report

In [ ]:
print('=' * 60)
print('  BRANDSPHERE AI — LOGO CNN CLASSIFIER SUMMARY')
print('=' * 60)
print(f'  Dataset:       Logo Dataset (Kaggle)')
print(f'  Classes used:  {TOP_N_CLASSES} (of 2341 total)')
print(f'  Images used:   {len(X_train)+len(X_val)+len(X_test)}')
print(f'  Image size:    {IMG_SIZE}x{IMG_SIZE}x3')
print(f'  Architecture:  Conv2D x4 → Flatten → Dense(512) → Softmax')
print(f'  Embedding dim: 512')
print(f'  Test Accuracy: {test_acc*100:.2f}%')
print(f'  Test Loss:     {test_loss:.4f}')
print(f'  Similarity:    Cosine similarity on 512-dim embeddings')
print(f'  Retrieval:     Top-5 most similar logos per query')
print('=' * 60)
print('  OUTPUTS SAVED:')
print('  - logo_cnn_model.h5          → full classifier')
print('  - logo_embedding_model.h5    → similarity engine')
print('  - logo_retrieval_package.pkl → Streamlit ready')
print('  - training_curves.png        → accuracy/loss plots')
print('  - pca_clusters.png           → embedding clusters')
print('  - top5_similar_logos.png     → retrieval demo')
print('=' * 60)